# 00 · 시작하기: 소리를 숫자로 보기

이 수업의 모델은 모두 **오디오를 입력으로 받아 기호(MIDI, 비트, 구간) 또는 다른 오디오를 출력**합니다. 모델을 돌리기 전에, 오디오가 컴퓨터 안에서 어떤 모양인지 먼저 봅니다.

- 소요 시간: 5분 · GPU 불필요
- 이 노트북에서 익히는 것: 준비 셀, 예시 음원·업로드·YouTube 중 고르기, 파형과 스펙트로그램, 재생 위젯
- 이후 노트북(01~07)은 모두 같은 "0. 준비" 셀로 시작합니다. 모델 가중치는 각 노트북이 처음 실행될 때 `models` 폴더에 내려받습니다.

In [ ]:
#@title 0. 준비: 수업 자료 받기와 모델 폴더 정하기 (실행만 하면 됩니다)
#@markdown 모델 가중치는 모두 `models` 폴더 하나에 받습니다. 기본 위치는 `/content`이고 런타임을 끄면 지워집니다. 체크하면 Google Drive의 `CT-AI_Microdegree/` 아래에 `models`와 `outputs`를 두어 다음 실행 때 다시 받지 않고, 다른 노트북의 결과도 이어서 쓸 수 있습니다.
KEEP_IN_DRIVE = False  #@param {type:"boolean"}
import os, sys, glob, time, json, subprocess
_cands = [os.environ.get('CTAI_REPO_DIR'), os.path.abspath(os.path.join(os.getcwd(), '..', '..')), '/content/CT-AI_Microdegree']
REPO = next((c for c in _cands if c and os.path.isfile(os.path.join(c, 'colab', 'ctai.py'))), None)
if REPO is None:
    REPO = '/content/CT-AI_Microdegree'
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/dabinkim0/CT-AI_Microdegree', REPO], check=True)
sys.path.insert(0, os.path.join(REPO, 'colab'))
REQ = os.path.join(REPO, 'colab', 'requirements')   # 노트북별 pip 패키지 목록
import ctai
P = ctai.setup(keep_in_drive=KEEP_IN_DRIVE)
AUDIO_DIR, ASSET_DIR, MODELS_DIR, CODE_DIR, OUT_DIR, IN_COLAB = P['AUDIO_DIR'], P['ASSET_DIR'], P['MODELS_DIR'], P['CODE_DIR'], P['OUT_DIR'], P['IN_COLAB']
print('예시 음원:', sorted(os.path.basename(p) for p in glob.glob(os.path.join(AUDIO_DIR, '*.wav'))))


In [ ]:
#@title 오디오 준비: 예시 음원 / 업로드 / YouTube 중 선택
import ipywidgets as W, soundfile as sf, numpy as np, librosa
from IPython.display import Audio, display
import matplotlib.pyplot as plt

EXAMPLES = {'밴드 (Let\'s Go Fishin\', 보컬+밴드)': 'band_lets_go_fishin_60s.wav', '재즈 (Vibe Ace)': 'jazz_vibe_ace_60s.wav', '피아노 (Scarlatti K.455)': 'piano_scarlatti_k455_30s.wav', '트럼펫 솔로 (5초)': 'trumpet_solo_5s.wav'}
SOURCE = '예시'      #@param ["예시", "업로드", "YouTube"]
EXAMPLE = "밴드 (Let's Go Fishin', 보컬+밴드)"   #@param ["밴드 (Let's Go Fishin', 보컬+밴드)", "재즈 (Vibe Ace)", "피아노 (Scarlatti K.455)", "트럼펫 솔로 (5초)"]
YOUTUBE_URL = ''   #@param {type:"string"}
MAX_SECONDS = 30   #@param {type:"integer"}
TARGET_SR = 22050

def load_audio(path, sr=TARGET_SR, max_sec=MAX_SECONDS, mono=True):
    y, _ = librosa.load(path, sr=sr, mono=mono)
    if max_sec: y = y[..., :int(max_sec * sr)]
    return y, sr

if SOURCE == '예시':
    audio_path = os.path.join(AUDIO_DIR, EXAMPLES[EXAMPLE])
elif SOURCE == '업로드':
    from google.colab import files
    up = files.upload(); audio_path = list(up.keys())[0]
else:
    # 수업 내 학습 목적으로만 사용하세요. 저작권이 있는 음원은 외부에 배포하지 않습니다.
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'yt-dlp'], check=True)
    subprocess.run(['yt-dlp', '-q', '-x', '--audio-format', 'wav', '-o', 'yt_input.%(ext)s', YOUTUBE_URL], check=True)
    audio_path = 'yt_input.wav'

y, sr = load_audio(audio_path)
mono = y if y.ndim == 1 else y.mean(0)
print('파일:', audio_path, '| 길이: %.1f s | sr=%d' % (len(mono) / sr, sr))
plt.figure(figsize=(14, 2)); plt.plot(np.arange(len(mono)) / sr, mono, lw=.4, color='#58c4dd'); plt.xlabel('time (s)'); plt.title('waveform'); plt.tight_layout(); plt.show()
display(Audio(mono, rate=sr))


## 1초에 22,050개의 숫자

위 파형은 1초에 22,050개(샘플링 레이트)의 숫자를 선으로 이은 것입니다. 모델은 이 숫자열을 직접 보거나, 아래처럼 **시간-주파수 그림(스펙트로그램)**으로 바꿔서 봅니다.

In [ ]:
#@title 스펙트로그램: 모델이 "보는" 그림
import librosa.display
S = librosa.amplitude_to_db(np.abs(librosa.stft(mono, n_fft=2048, hop_length=512)), ref=np.max)
M = librosa.power_to_db(librosa.feature.melspectrogram(y=mono, sr=sr, n_mels=128), ref=np.max)
fig, ax = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
librosa.display.specshow(S, sr=sr, hop_length=512, x_axis='time', y_axis='log', ax=ax[0], cmap='magma'); ax[0].set_title('STFT (log-frequency)')
librosa.display.specshow(M, sr=sr, x_axis='time', y_axis='mel', ax=ax[1], cmap='magma'); ax[1].set_title('mel-spectrogram (피아노 채보 모델의 입력)')
plt.tight_layout(); plt.show()
print('STFT 행렬 크기:', S.shape, '(주파수 bin × 시간 프레임)  | mel:', M.shape)

## 입력 → 출력으로 모델 나누기 (강의 슬라이드 THU 16)

| 유형 | 입력 → 출력 | 노트북 |
|---|---|---|
| A. 분석 | 피아노 오디오 → MIDI | 01 Online AMT |
| A. 분석 | 음악 오디오 → 멀티악기 MIDI | 02 YourMT3+ |
| A. 분석 | 음악 오디오 → BPM · 비트 · 구간 | 03 Allin1 |
| B. 편집 | 음악 오디오 → 6 스템 | 04 Demucs |
| B. 편집 | 단선율 오디오 → 단선율 오디오 | 05 Tone Transfer (DDSP) |
| C. 생성 | 텍스트 → 음악 오디오 | 06 Stable Audio 3 생성 |
| C. 생성 | 음악 오디오 + 텍스트 → 음악 오디오 | 07 Stable Audio 3 인페인팅 |

**해 보기**: `SOURCE`를 바꿔 자기 음원을 넣어 보세요. 스펙트로그램에서 드럼(세로 줄)과 보컬·악기(가로 줄)가 어떻게 다르게 보이나요?

In [ ]:
#@title 저장: 다듬은 음원을 outputs 폴더에 저장 (다른 노트북에서 재사용)
out = os.path.join(OUT_DIR, 'my_input.wav'); sf.write(out, mono, sr); print('저장:', out)